# 01 — PPO Training (3D Pack)

Runs `training/train_pack_ppo_3d.py` from the cloned repository.
No training code lives in this notebook.

- Algorithm: PPO (on-policy, VecNormalize)
- Pack: 4×3×2 cells, 7-element observation, continuous cooling action
- Full run: 3 000 000 steps (~1–2 h on T4 GPU)
- Checkpoints every 50 000 steps

> **Runtime → Change runtime type → T4 GPU** before running.

## 1 — Mount Drive and Clone / Pull Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/RL-Battery-Thermal-Management.git"
REPO_DIR = "/content/RL-Battery-Thermal-Management"

In [ ]:
import os

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

## 2 — Project Root and Dependencies

In [ ]:
%cd /content/RL-Battery-Thermal-Management
!pip install -q -r requirements.txt
print("Ready.")

## 3 — GPU Check

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU — training will be slow.")

## 4 — Output Paths (Drive)

In [ ]:
BASE = "/content/drive/MyDrive/battery_rl"

PPO_SAVE_DIR = f"{BASE}/models/ppo_3d_pack"
PPO_LOG_DIR  = f"{BASE}/logs/ppo_3d_pack"

os.makedirs(PPO_SAVE_DIR, exist_ok=True)
os.makedirs(PPO_LOG_DIR,  exist_ok=True)

print("Models →", PPO_SAVE_DIR)
print("Logs   →", PPO_LOG_DIR)

## 5 — Diagnostic

In [ ]:
!pwd
!find envs -maxdepth 2 -type f | sort
!find configs -maxdepth 2 -type f | sort
!grep -R "from envs" -n training
!grep -R "from configs" -n training

## 6 — Smoke Test (10 000 steps)

Run this first. If it completes without error, proceed to full training.

In [ ]:
!PYTHONPATH=. python training/train_pack_ppo_3d.py \
  --timesteps 10000 \
  --n-envs 2 \
  --save-dir {PPO_SAVE_DIR} \
  --log-dir  {PPO_LOG_DIR}

In [ ]:
print("Smoke test outputs:")
for f in sorted(os.listdir(PPO_SAVE_DIR)):
    print(" ", f)

## 7 — Full Training (3 000 000 steps)

Checkpoints save every 50 000 steps to `checkpoints/` inside `PPO_SAVE_DIR`.
If Colab disconnects, use the resume cell below.

In [ ]:
!PYTHONPATH=. python training/train_pack_ppo_3d.py \
  --timesteps 3000000 \
  --n-envs 8 \
  --save-dir {PPO_SAVE_DIR} \
  --log-dir  {PPO_LOG_DIR} \
  --device auto

## 8 — Resume from Checkpoint (if interrupted)

In [ ]:
ckpt_dir = os.path.join(PPO_SAVE_DIR, "checkpoints")
if os.path.isdir(ckpt_dir):
    ckpts = sorted(os.listdir(ckpt_dir))
    for c in ckpts:
        print(c)
    if ckpts:
        print("\nLatest:", os.path.join(ckpt_dir, ckpts[-1]))
else:
    print("No checkpoints yet.")

In [ ]:
# Set RESUME_PATH to the checkpoint you want to resume from, then uncomment.

# RESUME_PATH = f"{PPO_SAVE_DIR}/checkpoints/ppo_pack_checkpoint_XXXXXX_steps.zip"
#
# !PYTHONPATH=. python training/train_pack_ppo_3d.py \
#   --timesteps 3000000 \
#   --n-envs 8 \
#   --save-dir {PPO_SAVE_DIR} \
#   --log-dir  {PPO_LOG_DIR} \
#   --device auto \
#   --resume-from {RESUME_PATH}

## 9 — TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {PPO_LOG_DIR}

## 10 — Verify Saved Files

In [ ]:
print(f"Contents of {PPO_SAVE_DIR}:")
for root, dirs, files in os.walk(PPO_SAVE_DIR):
    for f in sorted(files):
        rel = os.path.relpath(os.path.join(root, f), PPO_SAVE_DIR)
        size_mb = os.path.getsize(os.path.join(root, f)) / 1e6
        print(f"  {rel}  ({size_mb:.1f} MB)")